## Классификация: превышает ли значение CC50 медианное значение выборки

In [1]:
!pip install catboost -q

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

RANDOM_STATE = 42

In [2]:
data = pd.read_csv('data/preprocessed_data.csv')

In [3]:
# Подготовим данные
ex_colums = ['log_IC50', 'log_CC50', 'log_SI', 'IC50, mM', 'CC50, mM', 'SI',
                'IC50_median', 'CC50_median', 'SI_median', 'SI_8']

X = data.drop(columns=ex_colums)
y = data['CC50_median']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          
    random_state=RANDOM_STATE
)
X_train.shape

(798, 30)

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipline = Pipeline([('scaler', 'passthrough'), ('models', LogisticRegression())])

In [5]:
# Параметры для различных моделей
param_grid = [
    {
        'scaler': [StandardScaler(), MinMaxScaler(), 'passthrough'],
        'models': [LogisticRegression(random_state=RANDOM_STATE)],
        'models__C': [0.1, 1, 10],
        'models__solver': ['saga', 'liblinear'],
        'models__penalty': ['l1', 'l2'],
    },
    {
        'scaler': ['passthrough'],
        'models': [RandomForestClassifier(random_state=RANDOM_STATE)],
        'models__n_estimators': [100, 150, 300],
        'models__max_depth': [None, 30, 40],
        'models__min_samples_split': [2, 5, 10],
        'models__min_samples_leaf': [1, 3, 5]
    },
    {
       'scaler': ['passthrough'],
       'models': [CatBoostClassifier(random_state=RANDOM_STATE, verbose=0)],
       'models__iterations': [100, 200, 500, 1000],
       'models__learning_rate': [0.01, 0.1],
       'models__depth': [5, 10, 15]
    },
    {
        'scaler': [StandardScaler(), MinMaxScaler()],
        'models': [KNeighborsClassifier()],
        'models__n_neighbors': [3, 5, 8],
        'models__weights': ['uniform', 'distance']
    }
]

In [6]:
# Для подбора гиперпараметров воспользуемся RandomizedSearchCV
random_search = RandomizedSearchCV(
    pipline,
    param_distributions=param_grid,
    n_iter=30,
    cv=cv,
    scoring='roc_auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [7]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scaler', 'passthrough'),
                                             ('models', LogisticRegression())]),
                   n_iter=30, n_jobs=-1,
                   param_distributions=[{'models': [LogisticRegression(random_state=42)],
                                         'models__C': [0.1, 1, 10],
                                         'models__penalty': ['l1', 'l2'],
                                         'models__solver': ['saga',
                                                            'liblinear'],
                                         'scale...
                                        {'models': [CatBoostClassifier(random_state=42, verbose=0)],
                                         'models__depth': [5, 10, 15],
                                         'models__iterations': [100, 200, 500,
                                                                1000],
                                         'models__learning_rate': [0.01, 0.1],
                                         'scaler': ['passthrough']},
                                        {'models': [KNeighborsClassifier()],
                                         'models__n_neighbors': [3, 5, 8],
                                         'models__weights': ['uniform',
                                                             'distance'],
                                         'scaler': [StandardScaler(),
                                                    MinMaxScaler()]}],
                   random_state=42, scoring='roc_auc', verbose=1)

In [8]:
print('Лучшая модель и её параметры:\n\n', random_search.best_params_) 
print('Метрика  для лучшей модели:\n', random_search.best_score_) 

Лучшая модель и её параметры:

 {'scaler': 'passthrough', 'models__learning_rate': 0.01, 'models__iterations': 1000, 'models__depth': 5, 'models': CatBoostClassifier(random_state=42, verbose=0)}
Метрика  для лучшей модели:
 0.8478978574181142


In [9]:
best_model = random_search.best_estimator_

# Оценим на тестовой выборке, посмотрим на несколько метрик
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f'accuracy: {accuracy}')
print(f'f1: {f1}')
print(f'roc_auc: {roc_auc}')

accuracy: 0.775
f1: 0.7867298578199052
roc_auc: 0.854185508994071


In [10]:
# Сравнение метрик моделей
res = pd.DataFrame(random_search.cv_results_)
res['model'] = res['params'].apply(lambda x: type(x['models']).__name__)
res['roc_auc'] = res['mean_test_score']
res.groupby('model')['roc_auc'].agg(['mean', 'min', 'max'])

,mean,min,max
model,,,
CatBoostClassifier,0.841122,0.825679,0.847898
LogisticRegression,0.757931,0.649349,0.774720
RandomForestClassifier,0.842876,0.834793,0.846866


В ходе выполнения задачи классификации (превышает ли значение CC50 медианное значение выборки) рассмотрены  несколько моделей с разными гиперпараметрами (LogisticRegression(), RandomForestClassifier(), CatBoostClassifier(), KNeighborsClassifier()). Используя RandomizedSearchCV следующая модель была определена как лучшая:  {'scaler': 'passthrough', 'models__learning_rate': 0.01, 'models__iterations': 1000, 'models__depth': 5, 'models': CatBoostClassifier(random_state=42, verbose=0)}. 

Метрика:

accuracy: 0.775
f1: 0.7867298578199052
roc_auc: 0.854185508994071

CatBoost и RF показали близкие результаты, логистическая регрессия им уступила. f1 = 0.79 показывает, что есть хороший баланс между точностью и полнотой.

В качестве рекомендации по улучшение можно попробовать применить стекинг лучших моделей, провести анализ важности признаков, чтобы выделить наиболее влияющие.